<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c42-header.png" alt="Nextia Learning · Vision AI: See, Count and Inspect" width="100%">

# Labels: how many, and how good

**[Labels: how many, and how good](https://learning.nextia-ai.com/courses/vision/m03/labels-how-many-and-how-good/)** · Vision AI: See, Count and Inspect · Module 3, lesson 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** measure the agreement of two labellers, find training photos with a wrong label with confident learning, fix them, measure the change, and find photos that are in the catalogue twice.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells. No model download, no account and no key. |
| **You should know** | Linear probes, from [Linear probes and few-shot](https://learning.nextia-ai.com/courses/vision/m03/linear-probes-and-few-shot/), and cross-validation, from [Train, validation and test sets](https://learning.nextia-ai.com/courses/ml/m03/train-validation-and-test-sets/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, Pillow 13.0.0, scikit-learn 1.9.1 and matplotlib 3.11.2, cleanlab 2.9.0 (PyTorch 2.15.0 and timm 1.0.31 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/vision/m03/labels-how-many-and-how-good/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C42/M03-L04-labels-how-many-and-how-good/labels-how-many-and-how-good-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, pillow 13.0.0, scikit-learn 1.9.1, scikit-image 0.26.0, cleanlab 2.9.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 pillow==13.0.0 scikit-learn==1.9.1 scikit-image==0.26.0 matplotlib==3.11.2 cleanlab==2.9.0
import warnings; warnings.filterwarnings("ignore")
import numpy; print("numpy", numpy.__version__)
import pandas; print("pandas", pandas.__version__)
import PIL; print("pillow", PIL.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import skimage; print("scikit-image", skimage.__version__)
import cleanlab; print("cleanlab", cleanlab.__version__)

## Setup: get the Larkfield images

Run the next cell. It downloads the image sets that this lesson needs (about 19 MB, as zip files) into a folder `larkfield-images`, with the course's helper `compare.py`, checks each file's checksum and unpacks it. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-images` (in Colab: the **Files** panel on the left), then run the cell again.

Where the photos come from: the catalogue photos are cut from Open Images photos (CC BY 2.0: the author of each photo is in `data/catalogue/manifest.csv`), the returned candles are photos of the VisA inspection set (CC BY 4.0), and the pallets are drawn by our code. Read `data/dataset-card.md` for the details.

In [ ]:
# Setup: download the image sets into larkfield-images/, check them and unpack them.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C42/"
FILES = {
    "archives/catalogue.zip": "83752b3ce15a1b99f3b9201cc979769d7abe39633754a84865d7e60f0b3d7814",
    "archives/extras.zip": "f121137bc765866c25929a4233c557ba72b64eea0c806070cc04a86c28ec27ec",
    "compare.py": "65e58d2fe043105e3ff09626861a8f64e5548039df760ea10d6fc02c71f8fec3",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-images").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-images and run again.")
    if name.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(PROJECT / "data")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
n = sum(1 for p in Path("data").rglob("*") if p.is_file())
print(f"Ready: {n} files in larkfield-images/data.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, Pillow's `Image`, matplotlib, and the course helper as `cp` (the comparison table). It defines `show(paths)`, which draws photos in a row, and the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: images, the comparison table, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import compare as cp
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
CLASSES = ["bowl", "candle", "chair", "cushion", "lamp", "mug", "plant_pot", "vase"]

def show(paths, titles=None, size=2.2):
    """Draw photos in a row (at most 8)."""
    paths = list(paths)[:8]
    fig, axes = plt.subplots(1, len(paths), figsize=(size * len(paths), size))
    for i, (ax, p) in enumerate(zip(np.atleast_1d(axes), paths)):
        ax.imshow(Image.open(p)); ax.axis("off")
        if titles:
            ax.set_title(str(titles[i]), fontsize=8)
    plt.show()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Two labellers, twelve photos

The labels below are **constructed** for the lesson: two people labelled the same 12 photos as plant pot or vase.

**Percent agreement** is the share of photos where they agree. **Cohen's kappa** removes the agreement that two people would reach by chance: 1 is perfect, 0 is chance level.

In [ ]:
labeller_a = ["plant pot", "plant pot", "vase", "vase", "plant pot", "vase", "plant pot", "vase", "plant pot", "vase", "plant pot", "vase"]
labeller_b = ["plant pot", "vase",      "vase", "vase", "plant pot", "vase", "plant pot", "plant pot", "plant pot", "vase", "plant pot", "vase"]
from sklearn.metrics import cohen_kappa_score
agree = np.mean([a == b for a, b in zip(labeller_a, labeller_b)])
print("agreement", round(float(agree), 3), " kappa", round(cohen_kappa_score(labeller_a, labeller_b), 3))

> **Check.** Agreement 0.833 (10 of 12), kappa 0.667.

## 2. Find suspect labels with confident learning

For each training photo, a probe that did **not** see that photo gives a probability to each class (5-fold cross-validation). A photo whose own label gets a low probability is a **suspect**. cleanlab ranks them.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from cleanlab.filter import find_label_issues
catalogue = pd.read_csv("data/catalogue/manifest.csv")
train = catalogue[catalogue["split"] == "train"].reset_index(drop=True)
test = catalogue[catalogue["split"] == "test"].reset_index(drop=True)

def embeddings(kind, ids):
    z = np.load(f"data/embeddings/{kind}.npz")
    X = np.stack([z[i] for i in ids]).astype(np.float32)
    return X / np.linalg.norm(X, axis=1, keepdims=True)

Xtrain, Xtest = embeddings("clip", train["image_id"]), embeddings("clip", test["image_id"])

def suspects(labels):
    """Confident learning: out-of-fold probabilities of a probe, then cleanlab's ranked list of label issues."""
    y = np.array([CLASSES.index(k) for k in labels])
    P = cross_val_predict(LogisticRegression(C=2.0, max_iter=5000), Xtrain, y, cv=5, method="predict_proba", n_jobs=1)
    issues = find_label_issues(y, P, return_indices_ranked_by="self_confidence", n_jobs=1)
    return pd.DataFrame({"image_id": train["image_id"][issues].values, "label": [labels[i] for i in issues],
                         "the model says": [CLASSES[P[i].argmax()] for i in issues],
                         "probability of the label": P[issues, y[issues]].round(3)})

found = suspects(list(train["label"]))
print(len(found), "suspects")
found.head(10)

> **Check.** 11 suspects. Most of the first ten are plant pots and vases.

> **Predict.** How many of the first ten are real errors?

In [ ]:
top = found.head(10)
paths = ["data/" + train.set_index("image_id").loc[i, "file"] for i in top["image_id"]]
show(paths[:5], [f"{i}\n{l}" for i, l in zip(top["image_id"][:5], top["label"][:5])])
show(paths[5:], [f"{i}\n{l}" for i, l in zip(top["image_id"][5:], top["label"][5:])])

## 3. Your turn: fix the vase labels

> **Your turn.** Look at the photos labelled **vase** in the top 10. Write the IDs of those that clearly show a plant growing in soil in a pot in the list `fixed_ids`, sorted. Hint: there are four. Two other vase photos are hard cases (a glass globe with an air plant, a garden urn with leaves): keep their labels, and write them down for the labelling guide.

In [ ]:
# Look at the photos above. Write the IDs of the vase-labelled photos that show a plant pot:
fixed_ids = ...

In [ ]:
expect_hash('fixed_ids', None if fixed_ids is ... else sorted(fixed_ids), 'e497980db95a0e46')

The sealed key holds the label errors that the course planted. It is not secret: it is there to score your search.

In [ ]:
key = cp.reveal("data/sealed-key.txt")
planted = {e["image_id"] for e in key["label_errors"]}
print("planted errors in the top 10:", sum(i in planted for i in top["image_id"]), "of", len(planted))

> **Check.** 4 of 4.

## 4. Measure the change

Train the probe again with the fixed labels, and run the search again.

> **Predict.** Will the test accuracy go up by much?

In [ ]:
fixed = list(train["label"])
for i in (fixed_ids if fixed_ids is not ... else []):
    fixed[train["image_id"].tolist().index(i)] = "plant_pot"
for name, labels in (("labels as given", list(train["label"])), ("labels fixed", fixed)):
    model = LogisticRegression(C=2.0, max_iter=5000).fit(Xtrain, labels)
    print(name, "- test accuracy", round(float(np.mean(model.predict(Xtest) == test["label"].values)), 3),
          "- suspects", len(suspects(labels)))

> **Check.** Test accuracy 0.894 before and 0.894 after. Suspects: 11 before and 5 after. Four wrong labels among 453 do not move the test score. They are still worth fixing: the next model, the validation set and the people who read the labels all use them.

## 5. Photos that are in the catalogue twice

A photo uploaded twice, a little cropped or brighter, has an embedding almost the same as the first one. The next cell lists the 10 most similar pairs of training photos.

In [ ]:
S = Xtrain @ Xtrain.T
np.fill_diagonal(S, -1)
rows, cols = np.triu_indices(len(S), 1)
order = np.argsort(-S[rows, cols], kind="mergesort")[:10]
pairs = pd.DataFrame({"a": train["image_id"].values[rows[order]], "b": train["image_id"].values[cols[order]],
                      "cosine": S[rows[order], cols[order]].round(3)})
same = {(d["image_id"], d["same_as"]) for d in key["duplicates"]}
pairs["planted"] = [(a, b) in same or (b, a) in same for a, b in zip(pairs["a"], pairs["b"])]
pairs

> **Check.** 5 of the 6 planted second uploads are in the 10 most similar pairs.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/vision/m03/labels-how-many-and-how-good/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Boxes and IoU](https://learning.nextia-ai.com/courses/vision/m04/boxes-and-iou/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/vision/m03/labels-how-many-and-how-good/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The photos are open photos (each with its author and licence in the manifests) or images made by our code; no photo shows a real person.